### Building a RAG System with Chroma and langchain

### Introduction
Retrieval-Augmented Generation (RAG) is a powerful technique that combines the capabilities of large language models with external knowledge retrieval. This notebook will walk you through building a complete RAG system using:

- LangChain: A framework for developing applications powered by language models
- ChromaDB: An open-source vector database for storing and retrieving embeddings
- OpenAI: For embeddings and language model (you can substitute with other providers)



In [3]:
import os
from dotenv import load_dotenv
load_dotenv()


True

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

from typing import List
import numpy as np

In [4]:
print("""
RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge
""")


RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge



### Sample Data

In [5]:
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]

sample_docs

['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective f

In [7]:
import tempfile
temp_dir = tempfile.mkdtemp()

for i, doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i+1}.txt", 'w') as f:
        f.write(doc)

print(f"Sample docs created!! in {temp_dir}")        

Sample docs created!! in /var/folders/nv/3g6f6py91psc7jgvpcl6t6980000gn/T/tmp0sqw4ly5


In [9]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

dir_load = DirectoryLoader(temp_dir, glob="*.txt", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
documents = dir_load.load()

print(f"Documents Loaded: {len(documents)}")
print(documents[0].page_content[:100])

Documents Loaded: 3

    Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction bet


### Document Splitter


In [15]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50, length_function=len, separators=["\n\n", "\n", " ", ""])
chunks = text_splitter.split_documents(documents)

print(f"Length od chunks, {len(chunks)}")
for idx, chunk in enumerate(chunks):
    print(f"Chunk {idx+1}: {chunk.page_content[:100]}")
    print(f"Metadata {idx+1}: {chunk.metadata}")

Length od chunks, 7
Chunk 1: Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between 
Metadata 1: {'source': '/var/folders/nv/3g6f6py91psc7jgvpcl6t6980000gn/T/tmp0sqw4ly5/doc_3.txt'}
Chunk 2: Deep Learning and Neural Networks
Metadata 2: {'source': '/var/folders/nv/3g6f6py91psc7jgvpcl6t6980000gn/T/tmp0sqw4ly5/doc_2.txt'}
Chunk 3: Deep learning is a subset of machine learning based on artificial neural networks. 
    These networ
Metadata 3: {'source': '/var/folders/nv/3g6f6py91psc7jgvpcl6t6980000gn/T/tmp0sqw4ly5/doc_2.txt'}
Chunk 4: excel at sequential data processing.
Metadata 4: {'source': '/var/folders/nv/3g6f6py91psc7jgvpcl6t6980000gn/T/tmp0sqw4ly5/doc_2.txt'}
Chunk 5: Machine Learning Fundamentals
Metadata 5: {'source': '/var/folders/nv/3g6f6py91psc7jgvpcl6t6980000gn/T/tmp0sqw4ly5/doc_1.txt'}
Chunk 6: Machine learning is a subset of artificial intelligence that enables systems to learn 
    and impro
Metadata 6: {'source': '/var/folders/

### Embedding Models


In [ ]:
os.environ["OPEN_API_KEY"] = os.getenv("OPEN_API_KEY")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small") 
vector = embeddings.embed_query("sample text")
vector


### Initialize chroma DB vector and store vector embeddings in it

In [18]:
persist_directory = "./chroma_db"

vectorstore = Chroma.from_documents(
    documents=chunks, 
    embedding=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2'), collection_name="rag_collection", 
    persist_directory=persist_directory
)

print(f"Vector store created with number of vectors: {vectorstore._collection.count()}")
print(f"Persisted to: {persist_directory}")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10722.59it/s]


Vector store created with number of vectors: 7
Persisted to: ./chroma_db


In [21]:
query = "What is Machine Learning?"

similarity = vectorstore.similarity_search(query,k=3)
similarity

[Document(metadata={'source': '/var/folders/nv/3g6f6py91psc7jgvpcl6t6980000gn/T/tmp0sqw4ly5/doc_1.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': '/var/folders/nv/3g6f6py91psc7jgvpcl6t6980000gn/T/tmp0sqw4ly5/doc_1.txt'}, page_content='Machine Learning Fundamentals'),
 Document(metadata={'source': '/var/folders/nv/3g6f6py91psc7jgvpcl6t6980000gn/T/tmp0sqw4ly5/doc_2.txt'}, page_content='Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers

In [22]:
query = "What is NLP?"

similarity = vectorstore.similarity_search(query,k=3)
similarity

[Document(metadata={'source': '/var/folders/nv/3g6f6py91psc7jgvpcl6t6980000gn/T/tmp0sqw4ly5/doc_3.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
 Document(metadata={'source': '/var/folders/nv/3g6f6py91psc7jgvpcl6t6980000gn/T/tmp0sqw4ly5/doc_1.txt'}, page_content='Machine Learning Fundamentals'),
 Document(metadata={'source': '/var/folders/nv/3g6f6py91psc7jgvpcl6t6980000gn/T/tmp0sqw4ly5/doc_2.txt'}, page_content='Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brai

In [23]:
print(f"Query: {query}")
print(f"\nTop {len(similarity)} similar chunks:")
for i, doc in enumerate(similarity):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:200] + "...")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")

Query: What is NLP?

Top 3 similar chunks:

--- Chunk 1 ---
Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recogn...
Source: /var/folders/nv/3g6f6py91psc7jgvpcl6t6980000gn/T/tmp0sqw4ly5/doc_3.txt

--- Chunk 2 ---
Machine Learning Fundamentals...
Source: /var/folders/nv/3g6f6py91psc7jgvpcl6t6980000gn/T/tmp0sqw4ly5/doc_1.txt

--- Chunk 3 ---
Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning...
Source: /var/folders/nv/3g6f6py91psc7jgvpcl6t6980000gn/T/tmp0sqw4ly5/doc_2.txt


In [24]:
similar_score = vectorstore.similarity_search_with_score(query, k=3)
similar_score

[(Document(metadata={'source': '/var/folders/nv/3g6f6py91psc7jgvpcl6t6980000gn/T/tmp0sqw4ly5/doc_3.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
  0.6382690072059631),
 (Document(metadata={'source': '/var/folders/nv/3g6f6py91psc7jgvpcl6t6980000gn/T/tmp0sqw4ly5/doc_1.txt'}, page_content='Machine Learning Fundamentals'),
  1.577245831489563),
 (Document(metadata={'source': '/var/folders/nv/3g6f6py91psc7jgvpcl6t6980000gn/T/tmp0sqw4ly5/doc_2.txt'}, page_content='Deep learning is a subset of machine learning based on artificial neural networks. \n 

#### Understanding Similarity Scores
The similarity score represents how closely related a document chunk is to your query. The scoring depends on the distance metric used:

ChromaDB default: Uses L2 distance (Euclidean distance)

- Lower scores = MORE similar (closer in vector space)
- Score of 0 = identical vectors
- Typical range: 0 to 2 (but can be higher)


Cosine similarity (if configured):

- Higher scores = MORE similar
- Range: -1 to 1 (1 being identical)

### Initialize LLM, RAG Chain, Prompt Template, Query the RAG system

In [26]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI()
llm.invoke("What is Large Language Model?")

AIMessage(content="A Large Language Model (LLM) is a type of artificial intelligence system that uses deep learning techniques to understand and generate human language. These models are trained on vast amounts of text data to learn the patterns and structures of language, allowing them to perform tasks such as machine translation, text summarization, and speech recognition. Examples of LLMs include OpenAI's GPT-3 and Google's BERT.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 83, 'prompt_tokens': 13, 'total_tokens': 96, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DkSuJxhT8pJyayyj3IZdursxpDzsB', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc

In [27]:
from langchain.chat_models.base import init_chat_model

llm = init_chat_model("openai:gpt-3.5-turbo")
llm

ChatOpenAI(output_version=None, profile={'name': 'GPT-3.5-turbo', 'release_date': '2023-03-01', 'last_updated': '2023-11-06', 'open_weights': False, 'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'attachment': False, 'temperature': True, 'image_url_inputs': False, 'pdf_inputs': False, 'pdf_tool_message': False, 'image_tool_message': False, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x323818190>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x323818550>, root_client=<openai.OpenAI object at 0x3234bbed0>, root_async_client=<openai.AsyncOpenAI object at 0x3238182d0>, model_kwargs={}, openai_api_key=SecretStr('**********'), openai

### Modern RAG Chain

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain